# CYPHER | Benchmark Results

Compact notebook for the two benchmark figures intended for the project presentation.

**Evaluation context:** 130 evaluation samples and 30 reference samples. Results below are from the supplied synthetic benchmark. They are not evidence of real-world performance.


## Cell 1 — Load benchmark data

The supplied JSON results are embedded in this cell. Run it first. No external JSON file is required.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

benchmark = {'benchmark_metadata': {'timestamp': '2026-11-26T06:36:29Z', 'platform': 'Windows-11-10.0.26300-SP0', 'python_version': '3.12.14', 'evaluation_samples_count': 130, 'reference_samples_count': 30}, 'detector_benchmarks': {'duplicate_exact_sha256': {'description': 'Baseline exact byte SHA-256 duplicate detection', 'latency_ms': 0.23, 'peak_memory_kb': 13.55, 'exact_duplicates_only': {'tp': 5, 'fp': 5, 'tn': 40, 'fn': 0, 'precision': 0.5, 'recall': 1.0, 'f1': 0.6667, 'fpr': 0.1111, 'fnr': 0.0}, 'all_duplicates_including_recompressed': {'tp': 5, 'fp': 5, 'tn': 25, 'fn': 15, 'precision': 0.5, 'recall': 0.25, 'f1': 0.3333, 'fpr': 0.1667, 'fnr': 0.75}}, 'duplicate_perceptual_dhash': {'description': 'Perceptual difference hashing (dHash, max Hamming dist 8)', 'latency_ms': 112.91, 'peak_memory_kb': 126.75, 'binary_metrics': {'tp': 18, 'fp': 14, 'tn': 16, 'fn': 2, 'precision': 0.5625, 'recall': 0.9, 'f1': 0.6923, 'fpr': 0.4667, 'fnr': 0.1}, 'continuous_metrics': {'auroc': 0.7417, 'aupr': 0.5779}}, 'label_integrity_hungarian': {'description': 'Optimal bipartite matching (Hungarian IoU) for label integrity', 'latency_ms': 21.07, 'peak_memory_kb': 26.67, 'image_level_metrics': {'tp': 20, 'fp': 0, 'tn': 35, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'fpr': 0.0, 'fnr': 0.0}, 'object_level_breakdown': {'class_changes': 5, 'bbox_modifications': 0, 'missing_objects': 10, 'additional_objects': 10, 'matched_clean': 160}, 'reordered_annotations_false_positives': 0}, 'trigger_patch_entropy': {'description': 'Sliding window Shannon patch entropy', 'latency_ms': 286.91, 'peak_memory_kb': 318.18, 'binary_metrics': {'tp': 8, 'fp': 0, 'tn': 30, 'fn': 12, 'precision': 1.0, 'recall': 0.4, 'f1': 0.5714, 'fpr': 0.0, 'fnr': 0.6}, 'continuous_metrics': {'auroc': 0.87, 'aupr': 0.8635}}, 'poisoning_self_referenced_zscore': {'description': 'Baseline self-referenced RGB Z-score outlier detection', 'latency_ms': 1.97, 'peak_memory_kb': 31.7, 'binary_metrics': {'tp': 0, 'fp': 0, 'tn': 30, 'fn': 20, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'fpr': 0.0, 'fnr': 1.0}, 'continuous_metrics': {'auroc': 1.0, 'aupr': 1.0}}, 'poisoning_reference_calibrated_mad': {'description': 'Reference-calibrated robust Modified Z-score (Median/MAD)', 'latency_ms': 6.6, 'peak_memory_kb': 74.63, 'binary_metrics': {'tp': 20, 'fp': 8, 'tn': 22, 'fn': 0, 'precision': 0.7143, 'recall': 1.0, 'f1': 0.8333, 'fpr': 0.2667, 'fnr': 0.0}, 'continuous_metrics': {'auroc': 0.9867, 'aupr': 0.9767}}, 'ood_self_referenced_mahalanobis': {'description': 'Baseline self-referenced channel Mahalanobis distance', 'latency_ms': 3.31, 'peak_memory_kb': 22.83, 'binary_metrics': {'tp': 4, 'fp': 0, 'tn': 30, 'fn': 11, 'precision': 1.0, 'recall': 0.2667, 'f1': 0.4211, 'fpr': 0.0, 'fnr': 0.7333}, 'continuous_metrics': {'auroc': 0.6, 'aupr': 0.7203}}, 'ood_reference_calibrated_mahalanobis': {'description': 'Reference-calibrated Mahalanobis distance', 'latency_ms': 3.52, 'peak_memory_kb': 23.24, 'binary_metrics': {'tp': 9, 'fp': 4, 'tn': 26, 'fn': 6, 'precision': 0.6923, 'recall': 0.6, 'f1': 0.6429, 'fpr': 0.1333, 'fnr': 0.4}, 'continuous_metrics': {'auroc': 0.7956, 'aupr': 0.7829}}}}

meta = benchmark['benchmark_metadata']
records = []
for key, result in benchmark['detector_benchmarks'].items():
    if 'binary_metrics' in result:
        metrics = result['binary_metrics']
        scope = 'Binary detector metrics'
    elif 'image_level_metrics' in result:
        metrics = result['image_level_metrics']
        scope = 'Image-level label integrity'
    else:
        metrics = result['exact_duplicates_only']
        scope = 'Exact duplicates only'
    records.append({
        'key': key,
        'name': {
            'duplicate_exact_sha256': 'Exact SHA-256',
            'duplicate_perceptual_dhash': 'Perceptual dHash',
            'label_integrity_hungarian': 'Hungarian label matching',
            'trigger_patch_entropy': 'Trigger patch entropy',
            'poisoning_self_referenced_zscore': 'Self-referenced Z-score',
            'poisoning_reference_calibrated_mad': 'Reference-calibrated MAD',
            'ood_self_referenced_mahalanobis': 'Self-referenced Mahalanobis',
            'ood_reference_calibrated_mahalanobis': 'Reference-calibrated Mahalanobis'
        }[key],
        'f1': metrics['f1'],
        'scope': scope,
        'latency_ms': result['latency_ms'],
        'peak_memory_kb': result['peak_memory_kb']
    })
df = pd.DataFrame(records)
print(f"Evaluation samples: {meta['evaluation_samples_count']} | Reference samples: {meta['reference_samples_count']}")
print(f"Benchmark timestamp: {meta['timestamp']}")
display(df[['name', 'f1', 'scope', 'latency_ms', 'peak_memory_kb']].round(4))


## Cell 2 — Main figure: F1 score by detector

**This is the primary benchmark image to include in the LinkedIn post.** It displays the reported F1 score for each detector. The metric scope is not identical for every detector: SHA-256 is exact-duplicate-only, Hungarian matching is image-level, and the remaining methods use their reported binary metrics.


In [ ]:
plot_df = df.sort_values("f1", ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(plot_df["name"], plot_df["f1"])
ax.set_xlim(0, 1.12)
ax.set_xlabel("F1 score")
ax.set_title("CYPHER | Synthetic Benchmark: Detector F1 Comparison", pad=18)
ax.text(
    0, 1.015,
    f'{meta["evaluation_samples_count"]} evaluation samples | '
    f'{meta["reference_samples_count"]} reference samples',
    transform=ax.transAxes, fontsize=10, color="dimgray"
)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

for bar, value in zip(bars, plot_df["f1"]):
    ax.text(value + 0.015, bar.get_y() + bar.get_height()/2,
            f"{value:.3f}", va="center", fontsize=9)

fig.text(
    0.01, 0.01,
    "Metric scopes differ: exact-duplicate-only SHA-256; image-level Hungarian matching; "
    "binary metrics for other detectors. Synthetic benchmark only.",
    fontsize=8, color="dimgray"
)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("cypher_f1_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## Cell 3 — Supporting figure: baseline vs reference-calibrated methods

**This is the second benchmark image to include if you want two results figures.** It compares the reported F1 scores for the two baseline/reference-calibrated pairs. It is a descriptive comparison of the supplied results, not proof that calibration alone caused the change.


In [ ]:
pairs = [
    ("Poisoning detection", "poisoning_self_referenced_zscore",
     "poisoning_reference_calibrated_mad"),
    ("OOD detection", "ood_self_referenced_mahalanobis",
     "ood_reference_calibrated_mahalanobis")
]

labels = []
baseline = []
calibrated = []

for label, base_key, calibrated_key in pairs:
    base_row = df.loc[df["key"] == base_key].iloc[0]
    calibrated_row = df.loc[df["key"] == calibrated_key].iloc[0]
    labels.append(label)
    baseline.append(base_row["f1"])
    calibrated.append(calibrated_row["f1"])

x = np.arange(len(labels))
width = 0.34

fig, ax = plt.subplots(figsize=(9, 5.5))
b1 = ax.bar(x - width/2, baseline, width, label="Self-referenced baseline")
b2 = ax.bar(x + width/2, calibrated, width, label="Reference-calibrated")

ax.set_ylabel("F1 score")
ax.set_title("CYPHER | Baseline vs Reference-Calibrated F1")
ax.set_xticks(x, labels)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.025,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("cypher_calibration_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## Interpretation and limitations

- The Hungarian label matching F1 is image-level and must not be interpreted as object-level F1.
- SHA-256 uses the exact-duplicates-only result here. The separate recompressed-inclusive result is F1 0.3333 and should not be conflated with it.
- The self-referenced Z-score reports AUROC 1.0 but F1 0.0 at its reported operating threshold. AUROC measures ranking across thresholds, while F1 uses one operating threshold. Check score direction and threshold selection before making claims from that AUROC.
- These are supplied synthetic benchmark results, not real-world validation.
- The results represent one reported run. No repeated-run variance or confidence intervals are provided.
